# 04 — Clinical / Tabular Feature Extraction
**AnemiaFusionNet — Stage 4**

Trains/fine-tunes a tabular neural network (`ClinicalEncoder`) on clinical CBC features standalone on the GPU, and exports 256-d clinical patient embeddings for the multimodal fusion model.

## 4.1 Set Working Directory and Load Packages

In [3]:
import os
from pathlib import Path

# Change working directory to project root if executed from notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\files


## 4.2 Load Tabular Dataset and Prepare Preprocessing Pipeline
We load `clinical_pretrain.csv`, simulate categorical variables, fit `StandardScaler` and `OrdinalEncoder` pipelines, and split into train/val sets.

In [4]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split

clinical_df = pd.read_csv("data/processed/clinical_pretrain.csv")

# Simulate categorical variables matching the master dataset schema
np.random.seed(42)
clinical_df["Gender"] = np.random.choice(["M", "F"], size=len(clinical_df))
clinical_df["region_type"] = np.random.choice(["urban", "rural"], size=len(clinical_df))

# Define columns (we exclude HGB/hemoglobin to prevent target leakage in clinical encoder)
NUMERIC_COLS = ["WBC", "LYMp", "NEUTp", "LYMn", "NEUTn", "RBC", "HCT", "MCV", "MCH", "MCHC", "PLT", "PDW", "PCT"]
CATEGORICAL_COLS = ["Gender", "region_type"]

# Split data into train / val
train_df, val_df = train_test_split(
    clinical_df, 
    test_size=0.20, 
    stratify=clinical_df["label"], 
    random_state=42
)

# Fit scaling and encoding pipelines on the training set
num_scaler = StandardScaler()
num_scaler.fit(train_df[NUMERIC_COLS])

cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
cat_encoder.fit(train_df[CATEGORICAL_COLS])

# Save preprocessing models for notebooks 05 and 06
os.makedirs("data/processed", exist_ok=True)
with open("data/processed/preprocessors.pkl", "wb") as f:
    pickle.dump({"scaler": num_scaler, "encoder": cat_encoder}, f)

print("Scaler and category encoder successfully fit and serialized.")

Scaler and category encoder successfully fit and serialized.


## 4.3 Prepare Tabular Data Loaders

In [5]:
import torch
from torch.utils.data import DataLoader
from src.datasets import ClinicalDataset

def preprocess_df(df, scaler, encoder):
    x_num = scaler.transform(df[NUMERIC_COLS])
    x_cat = encoder.transform(df[CATEGORICAL_COLS])
    labels = df["label"].values
    return x_num, x_cat, labels

x_num_train, x_cat_train, y_train = preprocess_df(train_df, num_scaler, cat_encoder)
x_num_val, x_cat_val, y_val = preprocess_df(val_df, num_scaler, cat_encoder)

train_dataset = ClinicalDataset(x_num_train, x_cat_train, y_train)
val_dataset = ClinicalDataset(x_num_val, x_cat_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

Train size: 1024 | Val size: 257


## 4.4 Standalone Tabular Model Pretraining
We initialize and train the `ClinicalClassifier` on the GPU for 15 epochs, using early stopping.

In [6]:
import torch.nn as nn
from src.clinical_encoder import ClinicalClassifier
from src.utils import calculate_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", device)

    # Categorical cardialities: Gender: 2, region_type: 2
cat_cardinalities = [2, 2]
model = ClinicalClassifier(num_numeric=len(NUMERIC_COLS), cat_cardinalities=cat_cardinalities, embed_dim=256).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for x_num, x_cat, labels in loader:
        x_num, x_cat, labels = x_num.to(device), x_cat.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(x_num, x_cat)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x_num.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for x_num, x_cat, labels in loader:
        x_num, x_cat = x_num.to(device), x_cat.to(device)
        logits, _ = model(x_num, x_cat)
        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = logits.argmax(dim=-1)
        
        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return calculate_metrics(all_labels, all_preds, all_probs)

best_val_f1 = 0.0
epochs = 15

for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    metrics = evaluate(model, val_loader, device)
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Val F1: {metrics['f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
    
    if metrics["f1"] >= best_val_f1:
        best_val_f1 = metrics["f1"]
        os.makedirs("models", exist_ok=True)
        torch.save(model.state_dict(), "models/clinical_classifier_best.pt")

print("Clinical standalone pretraining completed. Best Val F1:", best_val_f1)

Training on device: cuda
Epoch 01 | Loss: 0.5671 | Val F1: 0.8103 | Acc: 0.7704
Epoch 02 | Loss: 0.4206 | Val F1: 0.8706 | Acc: 0.8288
Epoch 03 | Loss: 0.3604 | Val F1: 0.9107 | Acc: 0.8794
Epoch 04 | Loss: 0.2828 | Val F1: 0.9032 | Acc: 0.8833
Epoch 05 | Loss: 0.3276 | Val F1: 0.9415 | Acc: 0.9222
Epoch 06 | Loss: 0.2507 | Val F1: 0.9415 | Acc: 0.9222
Epoch 07 | Loss: 0.2356 | Val F1: 0.9366 | Acc: 0.9183
Epoch 08 | Loss: 0.1954 | Val F1: 0.9467 | Acc: 0.9300
Epoch 09 | Loss: 0.1863 | Val F1: 0.9547 | Acc: 0.9416
Epoch 10 | Loss: 0.1907 | Val F1: 0.9493 | Acc: 0.9339
Epoch 11 | Loss: 0.1949 | Val F1: 0.9512 | Acc: 0.9377
Epoch 12 | Loss: 0.1727 | Val F1: 0.9388 | Acc: 0.9183
Epoch 13 | Loss: 0.1772 | Val F1: 0.9458 | Acc: 0.9300
Epoch 14 | Loss: 0.1711 | Val F1: 0.9527 | Acc: 0.9377
Epoch 15 | Loss: 0.1635 | Val F1: 0.9419 | Acc: 0.9261
Clinical standalone pretraining completed. Best Val F1: 0.9546827794561934


## 4.5 Export Clinical Embeddings
We load the best pre-trained clinical classifier weights and extract the clinical embeddings for all patients in `master_dataset.csv` using the fitted preprocessors.

In [7]:
model.load_state_dict(torch.load("models/clinical_classifier_best.pt"))
model.eval()

master_df = pd.read_csv("data/processed/master_dataset.csv")
x_num_master, x_cat_master, _ = preprocess_df(master_df, num_scaler, cat_encoder)

master_dataset = ClinicalDataset(x_num_master, x_cat_master, master_df["label"].values)
master_loader = DataLoader(master_dataset, batch_size=32, shuffle=False)

embeddings = []
print("Exporting clinical embeddings...")
with torch.no_grad():
    for x_num, x_cat, _ in master_loader:
        x_num, x_cat = x_num.to(device), x_cat.to(device)
        _, emb = model(x_num, x_cat)
        embeddings.append(emb.cpu())
        
all_clinical_embeddings = torch.cat(embeddings, dim=0)
print("Exported clinical embeddings shape:", all_clinical_embeddings.shape)

os.makedirs("data/embeddings", exist_ok=True)
torch.save(all_clinical_embeddings, "data/embeddings/clinical_embeddings.pt")
print("Saved clinical embeddings to data/embeddings/")

Exporting clinical embeddings...
Exported clinical embeddings shape: torch.Size([217, 256])
Saved clinical embeddings to data/embeddings/
